# 01 — Dataset, contexto e EDA

**Tech Challenge Fase 1 — G17**  
**Responsável:** Luis Conrado

| Item | Detalhe |
|------|--------|
| Problema | Classificar tumor **maligno (M)** × **benigno (B)** |
| Dataset | Breast Cancer Wisconsin (Diagnostic) |
| Arquivo | `data/raw/data.csv` |

Este notebook entrega: contexto clínico, documentação das colunas, exploração dos dados, visualizações e insights iniciais para o relatório.

## 1. Contexto do problema

O câncer de mama está entre os **tipos de câncer** mais frequentes em mulheres. O **diagnóstico precoce** melhora o prognóstico e reduz a mortalidade.

Neste projeto, usamos features numéricas extraídas de imagens de **FNA** (*Fine Needle Aspiration* — aspiração por agulha fina) de uma massa mamária. Cada feature descreve características do **núcleo celular** (tamanho, textura, irregularidade da borda etc.).

O objetivo do modelo é apoiar a classificação **maligno (M) vs benigno (B)**.

> **Importante:** o sistema de IA **apoia** a decisão do profissional de saúde — **nunca a substitui**. Em contexto clínico, um falso negativo (não detectar um caso maligno) costuma ser mais grave que um falso positivo; por isso, nas etapas seguintes, o grupo deve priorizar o **recall da classe maligna** (capacidade de detectar os casos malignos).

## 2. Setup

Rode **Run All**, ou execute as células de cima para baixo. Esta célula precisa rodar com sucesso antes das demais.

In [ ]:
# --- Setup: importa bibliotecas e define caminhos do projeto ---
from pathlib import Path

import pandas as pd          # tabelas / CSV
import numpy as np           # calculos numericos
import matplotlib.pyplot as plt  # graficos base
import seaborn as sns        # graficos estatisticos

# estilo visual padrao dos graficos
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)

# caminhos relativos a pasta notebooks/
ROOT = Path('..').resolve()                      # raiz do repositorio
RAW_CSV = ROOT / 'data' / 'raw' / 'data.csv'     # dataset baixado do Kaggle
FIG_DIR = ROOT / 'reports' / 'figuras'           # onde salvar as imagens
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Raiz:', ROOT)
print('CSV:', RAW_CSV, '| existe:', RAW_CSV.exists())
print('Figuras:', FIG_DIR)
print('Libs OK')


## 3. Carregar o dataset

In [ ]:
# --- Carrega o CSV e remove coluna vazia tipica do Kaggle ---
from pathlib import Path
import pandas as pd

# garante caminho mesmo se a celula de Setup nao tiver sido executada
ROOT = Path('..').resolve()
RAW_CSV = ROOT / 'data' / 'raw' / 'data.csv'

df = pd.read_csv(RAW_CSV)  # dataframe principal

# no CSV do Kaggle costuma existir uma coluna vazia chamada Unnamed: 32
unnamed = [c for c in df.columns if str(c).startswith('Unnamed')]
if unnamed:
    df = df.drop(columns=unnamed)
    print('Removidas colunas vazias:', unnamed)

print(f'Shape: {df.shape[0]} linhas x {df.shape[1]} colunas')
df.head()  # mostra as primeiras linhas


## 4. Descrição das colunas

Fonte: [Kaggle — Breast Cancer Wisconsin](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data/data).

Há **10 medidas** do núcleo celular, cada uma em 3 formas: `*_mean`, `*_se` (erro padrão) e `*_worst` (média dos 3 piores valores).

In [ ]:
# --- Glossario das features (texto util para o relatorio) ---
glossario = pd.DataFrame(
    [
        {'coluna_ou_grupo': 'id', 'descricao': 'Identificador da amostra (nao usar no modelo)'},
        {'coluna_ou_grupo': 'diagnosis', 'descricao': 'Alvo: M = maligno, B = benigno'},
        {'coluna_ou_grupo': 'radius', 'descricao': 'Media das distancias do centro aos pontos do perimetro'},
        {'coluna_ou_grupo': 'texture', 'descricao': 'Desvio padrao dos valores de escala de cinza'},
        {'coluna_ou_grupo': 'perimeter', 'descricao': 'Perimetro da celula'},
        {'coluna_ou_grupo': 'area', 'descricao': 'Area da celula'},
        {'coluna_ou_grupo': 'smoothness', 'descricao': 'Variacao local nos comprimentos de raio'},
        {'coluna_ou_grupo': 'compactness', 'descricao': 'perimeter2 / area - 1.0'},
        {'coluna_ou_grupo': 'concavity', 'descricao': 'Severidade de porcoes concavas do contorno'},
        {'coluna_ou_grupo': 'concave points', 'descricao': 'Numero de porcoes concavas do contorno'},
        {'coluna_ou_grupo': 'symmetry', 'descricao': 'Simetria'},
        {'coluna_ou_grupo': 'fractal_dimension', 'descricao': 'Complexidade fractal da borda'},
    ]
)
glossario


In [ ]:
# --- Lista colunas e tipos de dados ---
print('Colunas do CSV:')
print(list(df.columns))
print()
df.info()  # tipos, nulos e uso de memoria


## 5. Qualidade dos dados

In [ ]:
# --- Qualidade: nulos, duplicatas e estatisticas descritivas ---
missing = df.isna().sum()
qualidade = pd.DataFrame(
    {'nulos': missing, 'pct_nulos': (missing / len(df) * 100).round(2)}
)
qualidade = qualidade[qualidade['nulos'] > 0]  # so colunas com algum nulo

print(f'Linhas duplicadas: {df.duplicated().sum()}')
if qualidade.empty:
    print('Nenhum valor nulo encontrado nas colunas restantes.')
else:
    display(qualidade)

df.describe().T  # media, desvio, min, max etc. (transposto para ler melhor)


## 6. Balanceamento da variável alvo (`diagnosis`)

In [ ]:
# --- Balanceamento da classe alvo (M vs B) ---
contagem = df['diagnosis'].value_counts()
percentual = (df['diagnosis'].value_counts(normalize=True) * 100).round(2)
balanco = pd.DataFrame({'contagem': contagem, 'percentual_%': percentual})
display(balanco)

# grafico de barras das classes
fig, ax = plt.subplots()
sns.countplot(
    data=df,
    x='diagnosis',
    order=['B', 'M'],
    hue='diagnosis',
    palette={'B': '#4C9F70', 'M': '#C44E52'},  # verde=benigno, vermelho=maligno
    legend=False,
    ax=ax,
)
ax.set_title('Distribuicao das classes (diagnosis)')
ax.set_xlabel('Diagnostico (B = benigno, M = maligno)')
ax.set_ylabel('Contagem')
fig.tight_layout()
fig.savefig(FIG_DIR / '01_balanceamento_classes.png', dpi=150)  # exporta para o relatorio
plt.show()
print('Salvo:', FIG_DIR / '01_balanceamento_classes.png')


## 7. Distribuições das features (`*_mean`)

In [ ]:
# --- Distribuicoes das principais features *_mean por diagnostico ---
feature_cols = [c for c in df.columns if c not in {'id', 'diagnosis'}]
mean_cols = [c for c in feature_cols if c.endswith('_mean')][:6]  # 6 primeiras para visualizacao
print('Features plotadas:', mean_cols)

n = len(mean_cols)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
axes = np.array(axes).ravel()

for i, col in enumerate(mean_cols):
    # histograma + densidade, separado por M e B
    sns.histplot(
        data=df,
        x=col,
        hue='diagnosis',
        kde=True,
        element='step',
        stat='density',
        common_norm=False,
        ax=axes[i],
        palette={'B': '#4C9F70', 'M': '#C44E52'},
    )
    axes[i].set_title(col)

# esconde eixos vazios (se sobrar espaco no grid)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribuicoes por diagnostico', y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / '02_distribuicoes_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo:', FIG_DIR / '02_distribuicoes_features.png')


## 8. Boxplots — medianas e outliers

In [ ]:
# --- Boxplots: compara mediana e outliers entre B e M ---
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
axes = np.array(axes).ravel()

for i, col in enumerate(mean_cols):
    sns.boxplot(
        data=df,
        x='diagnosis',
        y=col,
        order=['B', 'M'],
        hue='diagnosis',
        palette={'B': '#4C9F70', 'M': '#C44E52'},
        legend=False,
        ax=axes[i],
    )
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Boxplots por diagnostico', y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / '03_boxplots_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo:', FIG_DIR / '03_boxplots_features.png')


## 9. Correlação (visão preliminar)

> Análise profunda de multicolinearidade fica com a **Pessoa 2**. Aqui só um panorama para a EDA.

In [ ]:
# --- Correlacao preliminar entre features *_mean (visao da EDA) ---
# Obs.: analise profunda de multicolinearidade fica com a Pessoa 2
subset = [c for c in feature_cols if c.endswith('_mean')]
corr = df[subset].corr()  # matriz de correlacao de Pearson

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap='vlag', center=0, annot=False, ax=ax)
ax.set_title('Matriz de correlacao (features *_mean)')
fig.tight_layout()
fig.savefig(FIG_DIR / '04_correlacao_subset.png', dpi=150)
plt.show()
print('Salvo:', FIG_DIR / '04_correlacao_subset.png')


## 10. Insights iniciais (texto para o relatório)

- A base tem **569 amostras** e dezenas de features numéricas derivadas de FNA.
- O alvo `diagnosis` tem classes `B` e `M`; em geral há **mais casos benignos** (desbalanceamento leve).
- Qualidade boa: após remover coluna vazia, tipicamente **sem nulos** relevantes; `id` deve ser excluído na modelagem.
- Features de **tamanho** (`radius`, `perimeter`, `area`) e de **irregularidade** (`concavity`, `concave points`, `compactness`) tendem a ser **maiores em tumores malignos**.
- Há **correlação alta** entre features de tamanho — a Pessoa 2 deve tratar multicolinearidade no pré-processamento.

**Próximo passo do grupo:** Pessoa 2 — limpeza, encoding, split estratificado e pipeline.

In [ ]:
# --- Resumo automatico para colar no relatorio / video ---
print('=== RESUMO EDA ===')
print(f'Shape: {df.shape[0]} linhas x {df.shape[1]} colunas')
print(f'Features numericas (sem id/diagnosis): {len(feature_cols)}')
print('Balanceamento:')
print(df['diagnosis'].value_counts().to_string())
print('\nFiguras exportadas:')
for p in sorted(FIG_DIR.glob('0*.png')):
    print(' -', p.name)
